In [1]:
from grokkingexperiments.config import load_config, print_config
from grokkingexperiments.model import GrokkingModel
from grokkingexperiments.tasks import select_task
from grokkingexperiments.train import train_model
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import scienceplots

plt.style.use(['science','notebook'])

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} to train")

Using cuda to train


In [2]:
# Load the config
config = load_config("../../../../config.toml")
print_config(config)

Config
  model
    | actv_func       relu
    | embed_dim       128
    | hidden_dim      256
  train
    | epochs          15000
  task
    | type            group
    | name            Zn
    | n               64
    | split           0.4
  settings
    | seed            1337
    | print_progress  True


In [ ]:
# Attempt to grok n = 8,16,24,32,...,256
start = 8
end = 256
step = 8
n_values = np.arange(start,end+1, step)

# To store the experiment results
experiment_results = []

# Change max_epochs to 100_000 and disable progress printing
config.train.epochs = 100_000
config.settings.print_progress = False

for current_n in n_values:
    print(f"\n--- Starting Experiment: n = {current_n} ---")

    # Overrite base config template to use the current n
    config.task.n = current_n

    # Create the model
    model = GrokkingModel(config).to(device)

    # Create the task
    # see tasks/select_task.py to see how we select
    # see tasks/task.py for shared methods between operations and groups
    # see tasks/operations for impelmented operations
    # see tasks/groups for base group class and implemented groups
    task = select_task(config)

    # Generate the data
    x_train, y_train, x_test, y_test = task.generate_train_test_data()

    # Move data to the same device as the model
    x_train = x_train.to(device)
    y_train = y_train.to(device)
    x_test = x_test.to(device)
    y_test = y_test.to(device)

    # Train the model
    train_model(config, model, x_train, y_train, x_test, y_test)

    # Store the results
    row = {
      "n": current_n,
      "num_epochs": model.epochs[-1],
      "final_train_loss": model.training_losses[-1],
      "final_test_loss": model.test_losses[-1],
      "final_train_accuracy": model.training_accuracies[-1],
      "final_test_accuracy": model.test_accuracies[-1]
    }
    experiment_results.append(row)


    print(f"--- Finished Experiment: n = {current_n} ---\n")




--- Starting Experiment: n = 8 ---
--- Finished Experiment: n = 8 ---


--- Starting Experiment: n = 16 ---
--- Finished Experiment: n = 16 ---


--- Starting Experiment: n = 24 ---
--- Finished Experiment: n = 24 ---


--- Starting Experiment: n = 32 ---
--- Finished Experiment: n = 32 ---


--- Starting Experiment: n = 40 ---
--- Finished Experiment: n = 40 ---


--- Starting Experiment: n = 48 ---
--- Finished Experiment: n = 48 ---


--- Starting Experiment: n = 56 ---
--- Finished Experiment: n = 56 ---


--- Starting Experiment: n = 64 ---
--- Finished Experiment: n = 64 ---


--- Starting Experiment: n = 72 ---
--- Finished Experiment: n = 72 ---


--- Starting Experiment: n = 80 ---
--- Finished Experiment: n = 80 ---


--- Starting Experiment: n = 88 ---
--- Finished Experiment: n = 88 ---


--- Starting Experiment: n = 96 ---
--- Finished Experiment: n = 96 ---


--- Starting Experiment: n = 104 ---
--- Finished Experiment: n = 104 ---


--- Starting Experiment: n = 112 ---


In [ ]:
# Convert results to datafrane
df = pd.DataFrame(experiment_results)
df

In [ ]:
plt.plot(df['n'], df['num_epochs'])
plt.title('Number of epochs required to grok')
plt.xlabel("n")
plt.ylabel("num epochs")
plt.show()